# 🔬 Powerball ML — the signal hunt

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jamelski1/PBML/blob/claude/powerball-prediction-ml-fxhi8i/notebooks/experiments.ipynb)

This is our **research log**. The premise: *maybe* there's a faint, undiscovered
structure in Powerball draws — a 0.01% chance — and the right model / target /
feature set could surface it. We treat that seriously and search systematically.

**But we refuse to fool ourselves.** Every experiment is judged by a *permutation
test*: we train on the real data, then on many copies where the past→future link
is shuffled away (marginals preserved). A result only counts as a **signal** if it
beats that shuffled null with **p < 0.01** on a held-out target.

- Beats null significantly → 🚨 a real, reproducible edge. The 0.01% we're after.
- Ties the null → the model only learned the stationary distribution (or noise).

This makes "did it work?" a *measurement*. Let's go pull levers.

## 0 · Setup — pull the harness from the repo

In [ ]:
import os
if not os.path.exists("pbml"):
    !git clone -q -b claude/powerball-prediction-ml-fxhi8i https://github.com/jamelski1/PBML.git pbml
import sys; sys.path.insert(0, "pbml/src")

import numpy as np, pandas as pd, torch
import experiments as E, runner as R

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| GPU:", torch.cuda.get_device_name(0) if device.type=="cuda" else "—")

## 1 · Load the real data

Every Powerball draw since the Oct-2015 rules change. If `data.ny.gov` is ever
blocked, upload a CSV with `Draw Date` + `Winning Numbers` columns and point
`load_draws` at it.

In [ ]:
try:
    df = E.load_draws()
except Exception as e:
    print("auto-download failed:", e, "\nupload a CSV instead:")
    from google.colab import files
    up = files.upload()
    df = E.load_draws(next(iter(up)))
print(f"{len(df)} draws  ({df['Draw Date'].min().date()} → {df['Draw Date'].max().date()})")
df.tail(3)[E.WHITE_COLS + ['pb']]

## 2 · Round 1 — a broad sweep

Fire every model at every target, quick-and-cheap, to see if *anything* pokes
above the noise. This is reconnaissance: low epochs, few permutations. Nothing
here is conclusive — we're looking for candidates worth a deep dive.

**Read the `gap` column** (real − permuted). On random data it wobbles around 0.
Anything with a suspiciously large positive gap graduates to Round 3.

In [ ]:
MODELS  = ["freq", "mlp", "lstm", "gru", "tcn", "transformer"]
TARGETS = ["white_multihot", "pb", "sum_bucket", "odd_count", "high_count"]

rows = []
for t in TARGETS:
    for m in MODELS:
        cfg = {"model": m, "target": t, "window": 24}
        rows.append(R.run_experiment(df, cfg, device=device, epochs=25, verbose=True))
sweep = pd.DataFrame(rows).sort_values("gap", ascending=False)

In [ ]:
# Leaderboard by raw gap. Expect sum_bucket/odd_count/high_count to show the
# biggest 'real' numbers — but that's the marginal distribution, not prediction.
# Round 3's permutation test is what separates the two.
sweep[["model","target","metric","real","permuted","gap"]].head(12)

## 3 · Round 2 — feature engineering on the best framings

Give the models more to work with: recency ('gap' — draws since each ball last
appeared) and calendar features ('dow', 'month'). If draw timing or physical
ball fatigue left any fingerprint, this is where it would show.

In [ ]:
FEATURE_SETS = {
    "base":        (),
    "recency":     ("gap",),
    "calendar":    ("dow", "month"),
    "everything":  ("gap", "dow", "month"),
}
rows = []
for name, ex in FEATURE_SETS.items():
    for m in ["lstm", "tcn", "transformer"]:
        cfg = {"model": m, "target": "white_multihot", "window": 24, "extras": ex}
        r = R.run_experiment(df, cfg, device=device, epochs=25, verbose=False)
        r["features"] = name
        rows.append(r); print(f"{name:>11} / {m:<11} gap={r['gap']:+.4f}")
pd.DataFrame(rows).sort_values("gap", ascending=False)[["features","model","real","permuted","gap"]].head(8)

## 4 · Round 3 — the verdict: rigorous permutation tests

Take the most promising configs and put them through the **real referee**:
train once on true data, ~30 times on shuffled nulls, and compute a p-value.

**This is the moment of truth.** `p < 0.01` prints `<-- SIGNAL!`. Anything else
is the model tying with chance, exactly as probability predicts. Edit
`CANDIDATES` to promote whatever looked interesting above.

In [ ]:
CANDIDATES = [
    {"model": "transformer", "target": "white_multihot", "window": 24},
    {"model": "tcn",         "target": "white_multihot", "window": 24, "extras": ("gap",)},
    {"model": "lstm",        "target": "sum_bucket",     "window": 24},
    {"model": "gru",         "target": "odd_count",      "window": 16},
]
verdicts = [R.permutation_test(df, c, n_perms=30, device=device, epochs=40)
            for c in CANDIDATES]
pd.DataFrame(verdicts)[["model","target","real","null_mean","z","p_value","signal"]]

## 5 · How to keep hunting

The harness makes new experiments one line each. Levers to try:

- **New targets** — add to `make_target` in `src/experiments.py`
  (e.g. consecutive-pairs count, sum parity, specific-number recurrence)
- **New features** — add channels in `make_features` (jackpot size, machine/ball-set
  ID if you can find that data — that's the only lever with a *physical* mechanism)
- **New models** — drop a class into `src/runner.py` (XGBoost, N-gram/Markov, a
  bigger Transformer, an ensemble)
- **Hyperparameters** — `window`, `epochs`, `lr`

Rule of the lab: **every promising result goes through `permutation_test` before
we believe it.** If you ever see `p < 0.01` reproduce across seeds and hold on a
fresh test set — stop, tell me, and we'll design a confirmation study. That would
be genuinely extraordinary.

And if 200 experiments all tie the null? That's not failure — that's us
*measuring* the randomness of a physical system with real rigor, which is its own
kind of cool. 🎲